# Interacting with the Internet in Python

In the previous lecture, we downloaded our example data file by using Jupyter Notebooks and a shell command (`curl`) like this:

In [1]:
!curl https://data.seattle.gov/resource/v5tj-kqhc.csv -o parks.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 47966    0 47966    0     0  46735      0 --:--:--  0:00:01 --:--:-- 46750


This is also how we have downloaded files throughout this course, including an assignment in which we created a bash script to download the data for us.

However, Python can also make Internet requests to download or upload data. Why might we choose to use Python to download data instead of a bash script or `curl` command?

* Data might change frequently, requiring an application to re-fetch the data dynamically
* Data might not be provided in a single file; it might require **pagination** (many downloads) to fetch all the data
* Data might be in a non-data format, such as HTML, which needs to be parsed in order to be used
* Data might be continuous, a stream of content that we need to process over time
* We might not want to save data to files, just process it

While we *could* address all of these scenarios with bash - it is a complete programming language, after all - bash is really not suited to more complex processing. Instead, it is more appropriate to use a more ergonomic programming lanuage like Python


----------------------

## Making Internet requests

Python's built in `urllib.request` module ([link to documentation](https://docs.python.org/3/library/urllib.request.html)) is the most basic tool that we can use to make requests to the Internet. Python's documentation even has [a full guide](https://docs.python.org/3/howto/urllib2.html) for working with it. We start by importing the module:

In [2]:
import urllib.request

As always in Jupyter notebooks, we can learn more about the module with tab completion and the `?` character. Try it:

In [ ]:
urllib.request.

The most useful functions in the module will probably be:
* `urllib.request.urlretrieve`
* `urllib.request.urlopen`

#### `urllib.request` like curl

We can do the same thing we did with `curl` before and save the content to a file with `urlretrieve`:

In [3]:
# Define a variable to store the URL we want to download
url = 'https://data.seattle.gov/resource/v5tj-kqhc.csv'

file_name_to_save = 'parks_python.csv'

# Download the content of the URL to a particular file
urllib.request.urlretrieve(url, file_name_to_save)

# Examine the file to show that it worked
with open(file_name_to_save) as file:
    print(f'First line: {file.readline()}')

First line: "pmaid","locid","name","address","zip_code","x_coord","y_coord","location_1"



#### Other features

We won't dive into all the features offered by `urllib.request`, but it does support more complex Internet request configuration, including:

* headers (both request and response)
* response codes
* request content (aka `body`) for `POST`/`PUT` requests, like form submission or file uploads
* errors
* authentication
* timeouts

HOWEVER, using these features becomes a fairly heavy-weight and manual process. If you need to do more complex Internet operations in future courses or jobs, you should prefer the `requests` library.

## The `requests` library

[The `requests` library](https://requests.readthedocs.io/en/latest/) is a third-party package that has more user-friendly configuration of advanced options (and is even recommended by the official Python documentation!). You can download and use the library like this:

In [ ]:
!conda install -y requests

In [4]:
import requests

In [5]:
result = requests.get(url)
result.text.split('\n')[0]

'"pmaid","locid","name","address","zip_code","x_coord","y_coord","location_1"'

Note that this is different behavior from the `urllib.request.urlretrieve` function that we used before. Instead of saving to a file, we now have the response directly and can operate on it in Python without saving to a file.

The library also has a few useful utilities that make working with it easier, such as:

A json decoder, which translates JSON into a Python dictionary:

In [6]:
json_url = 'https://data.seattle.gov/resource/v5tj-kqhc.json'
result = requests.get(json_url)
result.json()

[{'pmaid': '281',
  'locid': '2545',
  'name': '12th and Howe Play Park',
  'address': '1200 W Howe St',
  'zip_code': '98119',
  'x_coord': '-122.372985',
  'y_coord': '47.636097',
  'location_1': {'latitude': '47.636097', 'longitude': '-122.372985'}},
 {'pmaid': '4159',
  'locid': '2387',
  'name': '12th Ave S Viewpoint',
  'address': '2821 12TH Ave S',
  'zip_code': '98144',
  'x_coord': '-122.317765',
  'y_coord': '47.577953',
  'location_1': {'latitude': '47.577953', 'longitude': '-122.317765'}},
 {'pmaid': '4467',
  'locid': '2382',
  'name': '12th Ave Square Park',
  'address': '564 12th Ave',
  'zip_code': '98122',
  'x_coord': '-122.316455',
  'y_coord': '47.607427',
  'location_1': {'latitude': '47.607427', 'longitude': '-122.316455'}},
 {'pmaid': '4010',
  'locid': '2546',
  'name': '14th Ave NW Boat Ramp',
  'address': '4400 14th Ave NW',
  'zip_code': '98107',
  'x_coord': '-122.373536',
  'y_coord': '47.660775',
  'location_1': {'latitude': '47.660775', 'longitude': '-122

Iterating through the resulting data line by line:

In [7]:
result = requests.get(url)
for line in result.iter_lines():
    print(line)

b'"pmaid","locid","name","address","zip_code","x_coord","y_coord","location_1"'
b'"390","2404","Pratt Park","1800 S Main St","98144","-122.307257","47.601407","(47.601407, -122.307257)"'
b'"281","2545","12th and Howe Play Park","1200 W Howe St","98119","-122.372985","47.636097","(47.636097, -122.372985)"'
b'"4159","2387","12th Ave S Viewpoint","2821 12TH Ave S","98144","-122.317765","47.577953","(47.577953, -122.317765)"'
b'"4467","2382","12th Ave Square Park","564 12th Ave","98122","-122.316455","47.607427","(47.607427, -122.316455)"'
b'"4010","2546","14th Ave NW Boat Ramp","4400 14th Ave NW","98107","-122.373536","47.660775","(47.660775, -122.373536)"'
b'"296","296","3001 E Madison","3001 E Madison St","98112","-122.293173","47.625169","(47.625169, -122.293173)"'
b'"1000001","0","32nd Ave W Boat Launch","32nd Ave W","98199","-122.398685","47.632345","(47.632345, -122.398685)"'
b'"3158","2378","6th Ave NW Pocket Park","606  NW 76th St","98117","-122.363566","47.684202","(47.684202, -1

This would probably be better written as:

In [8]:
for line in result.iter_lines():
    print(line.decode(result.encoding))

"pmaid","locid","name","address","zip_code","x_coord","y_coord","location_1"
"390","2404","Pratt Park","1800 S Main St","98144","-122.307257","47.601407","(47.601407, -122.307257)"
"281","2545","12th and Howe Play Park","1200 W Howe St","98119","-122.372985","47.636097","(47.636097, -122.372985)"
"4159","2387","12th Ave S Viewpoint","2821 12TH Ave S","98144","-122.317765","47.577953","(47.577953, -122.317765)"
"4467","2382","12th Ave Square Park","564 12th Ave","98122","-122.316455","47.607427","(47.607427, -122.316455)"
"4010","2546","14th Ave NW Boat Ramp","4400 14th Ave NW","98107","-122.373536","47.660775","(47.660775, -122.373536)"
"296","296","3001 E Madison","3001 E Madison St","98112","-122.293173","47.625169","(47.625169, -122.293173)"
"1000001","0","32nd Ave W Boat Launch","32nd Ave W","98199","-122.398685","47.632345","(47.632345, -122.398685)"
"3158","2378","6th Ave NW Pocket Park","606  NW 76th St","98117","-122.363566","47.684202","(47.684202, -122.363566)"
"4404","2533",

Note that without the `decode` piece, each line is by default a "bytes" object - raw sets of 0s and 1s that we need to translate into a string with some kind of encoding.

## Exercise

We'll work again with [the Seattle pet license data from Seattle's Open Data Program](https://data.seattle.gov/City-Administration/Seattle-Pet-Licenses/jguv-t9rb/about_data). In a previous week, we saw that we could only download 1000 rows at once, and they all pertained to cats - although there's more than 38k rows in the dataset! We need to load the data in batches ("pages") so that we can put it all in a single data file.

The URL that we'll use looks like this (with our DATA 598B app token embedded in it):

```
https://data.seattle.gov/api/v3/views/jguv-t9rb/query.csv?app_token=XmnHk9K8KFspJPLY7pf6uuuAz&pageSize=10&pageNumber=1
```

That will download the first "page" of 10 results. You can move from one page to another by changing the value of the last part of the url, ie `pageNumber=2` instead of `pageNumber=1`.

Write Python code that downloads the first 10 pages of size 10 and prints them out. Note that each page will have its own header row with the column names, and you'll only want to include that once (as part of the first page).

In [9]:
def download_pets():
    url_template = 'https://data.seattle.gov/api/v3/views/jguv-t9rb/query.csv?app_token=XmnHk9K8KFspJPLY7pf6uuuAz&pageSize=10&pageNumber='

    for page_number in range(1, 11):
        content = requests.get(f'{url_template}{page_number}').text.strip()

        if page_number == 1:
            # If it's the first page, print the whole content,
            # including the column names
            print(content)
        else:
            # Otherwise, pull off the first line by splitting on the newline
            lines = content.split('\n', maxsplit=1)
            print(lines[1])

download_pets()

":id",":version",":created_at",":updated_at","license_issue_date","license_number","animal_s_name","species","primary_breed","secondary_breed","zip_code"
"row-ezee-cy56~5eqa","rv-2ufm.2umb~2iex","2026-10-06T23:35:02.889Z","2026-10-06T23:35:02.889Z","2022-12-12T00:00:00.000","833669","Garbanzo","Cat","Domestic Shorthair",,"98115"
"row-tycc-as4s~c3a8","rv-37yt-7ppx-nqe3","2026-10-06T23:35:02.889Z","2026-10-06T23:35:02.889Z","2022-12-14T00:00:00.000","S125986","Mia","Cat","Domestic Longhair","Mix","98126"
"row-vhbw-jdvi~nu8w","rv-hqzn_qwur~yr4d","2026-10-06T23:35:02.889Z","2026-10-06T23:35:02.889Z","2023-02-02T00:00:00.000","8042328","Benito","Cat","American Shorthair","Mix","98112"
"row-h8uc_4xuz_dskp","rv-v6iz~kjg5_cg5n","2026-10-06T23:35:02.889Z","2026-10-06T23:35:02.889Z","2023-03-05T00:00:00.000","8029390","Rory","Cat","Domestic Shorthair","Mix","98116"
"row-xmss-fen4-yh2j","rv-e764_42p5~n3aa","2026-10-06T23:35:02.889Z","2026-10-06T23:35:02.889Z","2023-03-08T00:00:00.000","8026779","

### Exercise bonus:

Extend your solution to the exercise:
* Instead of printing, modify your code to save the data to a file called `pets.csv`.
* Instead of loading only the first 10 pages, keep making requests until you download the entire dataset. You can tell that you're done when you get no data rows from a request (ie only the header row). It can help to load pages of size 1000 instead of 10.
* Use a dictionary to set the url's query parameters (the stuff after the `?`) using the `requests` library ([here's the documentation](https://requests.readthedocs.io/en/latest/user/quickstart/#passing-parameters-in-urls)).

In [10]:
def download_pets_bonus():
    url = 'https://data.seattle.gov/api/v3/views/jguv-t9rb/query.csv'
    with open('pets.csv', 'w') as output_file:
        page_number = 0
        keep_reading = True
        while keep_reading:
            page_number = page_number + 1
            # With the requests library, we can also define parameters as a dictionary
            query_params = {
                "app_token": "XmnHk9K8KFspJPLY7pf6uuuAz",
                "pageSize": 1000,
                "pageNumber": page_number
            }
            content = requests.get(f'{url}', params=query_params).text.strip()

            if page_number == 1:
                # If it's the first page, print the whole content,
                # including the column names
                print(content, file=output_file)
            else:
                # Otherwise, pull off the first line by splitting on the newline
                lines = content.split('\n', maxsplit=1)
                if len(lines) > 1 and len(lines[1]) > 0:
                    # Print if we have data to print
                    print(lines[1], file=output_file)
                else:
                    # We don't have any data, it's the end of the dataset
                    keep_reading = False

download_pets_bonus()

-----------

## Addendum: Alternative ways to use the `urllib.request` module

#### `urllib.request` like a file

The `urllib.request` module also allows us to specify a URL and then download it with `urlopen`, like this:

In [11]:
# Open the URL just like a file
with urllib.request.urlopen(url) as response:
    
    # Fetch and decode the content
    content = response.read().decode("utf-8")
    
    # Examine the content
    lines = content.split('\n')
    print(f'First line: {lines[0]}')

First line: "pmaid","locid","name","address","zip_code","x_coord","y_coord","location_1"


Note a few things about the previous code:

* We've used the same `with ... as ...` construction as we did with files, and you can think of this in exactly the same way! We're "opening" a URL.
* Instead of a `File` object, the response is an [`HTTPResponse` object](https://docs.python.org/3/library/http.client.html#httpresponse-objects). But like a file, it has a [`read()` function](https://docs.python.org/3/library/http.client.html#http.client.HTTPResponse.read) to access the underlying data.
* One **difference** between HTTP responses and files: with files, we can specify the encoding (how we translate bytes into text) when we open the file, whereas HTTP responses are just pure bytes, and we need to decode them (translate them into a string) before we can interpret them as text.
    * What is a byte? A byte is eight bits. A bit is a single 0 or 1. So a byte would be something like `10110001`.

#### `urllib.request` with a temporary file

There's one more straightforward approach to using `urllib.request` to download content from the Internet: use a temporary file, which is cleaned up when Python exits. Then we can operate on it just like a file, but don't have to worry about deleting it.

In [12]:
# Import two more standard library modules
import shutil
import tempfile

# Create a temporary file that will be deleted after the block
with tempfile.NamedTemporaryFile() as tmp_file:

    with urllib.request.urlopen(url) as response:
        # Copy the content into the temporary file
        shutil.copyfileobj(response, tmp_file)

    # Do something with the file
    with open(tmp_file.name) as file:
        print(f'First line: {file.readline()}')

# Now the file has been cleaned up!

First line: "pmaid","locid","name","address","zip_code","x_coord","y_coord","location_1"

